# 12 — Validate MDRO mart

CRE and MDR *P. aeruginosa* are summarized separately from the conventional antibiogram. The denominator here is **distinct patients with the base organism in each cohort**, regardless of whether they also have an ESCE-R or CRE reporting category. A patient is counted once per cohort/base organism; the phenotype status is assessed separately.

### Structural checks

In [ ]:
%sql
select
    count(*) as mart_rows,

    count(
        distinct mdro_mart_key
    ) as unique_keys,

    sum(
        case
            when denominator_count
                 <> positive_count
                  + not_detected_count
                  + not_evaluable_count
                then 1
            else 0
        end
    ) as denominator_mismatches,

    sum(
        case
            when positive_count > denominator_count
                then 1
            else 0
        end
    ) as numerator_exceeds_denominator,

    min(phenotype_pct) as min_phenotype_pct,
    max(phenotype_pct) as max_phenotype_pct

from workspace.micro_dev.mart_mdro;

This checks unique mart keys, numerator and denominator reconciliation, and percentage bounds. The saved output for this structural query was not available in the uploaded notebook; rerun it before publishing results.

### Reconstruct patient-level MDRO counts

In [2]:
%sql
with denominator_patient_groups as (

    select
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group,

        max(
            case
                when ast_cre_status <> 'not_applicable'
                    then 1
                else 0
            end
        ) as cre_eligible,

        max(
            case
                when ast_mdr_p_aeruginosa_status <> 'not_applicable'
                    then 1
                else 0
            end
        ) as mdr_p_aeruginosa_eligible

    from workspace.micro_dev.int_reporting_culture_organism

    group by
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group
),

expected_denominators as (

    select
        source_version,
        cohort_id,
        organism_reporting_group as organism_base_name,
        'CRE' as mdro_phenotype,
        count(*) as expected_denominator_count

    from denominator_patient_groups

    where cre_eligible = 1

    group by
        source_version,
        cohort_id,
        organism_reporting_group

    union all

    select
        source_version,
        cohort_id,
        organism_reporting_group,
        'MDR Pseudomonas aeruginosa',
        count(*)

    from denominator_patient_groups

    where mdr_p_aeruginosa_eligible = 1

    group by
        source_version,
        cohort_id,
        organism_reporting_group
),

expected_numerators as (

    select
        source_version,
        cohort_id,
        organism_reporting_group as organism_base_name,

        case
            when reporting_phenotype = 'cre'
                then 'CRE'

            when reporting_phenotype = 'mdr_p_aeruginosa'
                then 'MDR Pseudomonas aeruginosa'
        end as mdro_phenotype,

        count(*) as expected_positive_count

    from workspace.micro_dev.int_first_reporting_isolate

    where reporting_phenotype in (
        'cre',
        'mdr_p_aeruginosa'
    )

    group by
        source_version,
        cohort_id,
        organism_reporting_group,
        reporting_phenotype
),

comparison as (

    select
        m.source_version,
        m.cohort_id,
        m.organism_base_name,
        m.mdro_phenotype,

        m.denominator_count,
        d.expected_denominator_count,

        m.positive_count,
        coalesce(
            n.expected_positive_count,
            0
        ) as expected_positive_count,

        m.not_detected_count,
        m.not_evaluable_count,
        m.phenotype_pct,

        case
            when m.denominator_count
                 <> d.expected_denominator_count
                then 'denominator_mismatch'

            when m.positive_count
                 <> coalesce(
                     n.expected_positive_count,
                     0
                 )
                then 'numerator_mismatch'

            else 'pass'
        end as validation_status

    from workspace.micro_dev.mart_mdro m

    left join expected_denominators d
        on m.source_version = d.source_version
       and m.cohort_id = d.cohort_id
       and m.organism_base_name = d.organism_base_name
       and m.mdro_phenotype = d.mdro_phenotype

    left join expected_numerators n
        on m.source_version = n.source_version
       and m.cohort_id = n.cohort_id
       and m.organism_base_name = n.organism_base_name
       and m.mdro_phenotype = n.mdro_phenotype

)

select
    validation_status,
    count(*) as mart_rows

from comparison

group by
    validation_status

order by
    validation_status

,validation_status,mart_rows
0,pass,31


I independently reconstruct the expected denominator and positive count from the intermediate reporting models and compare them with `mart_mdro`. The saved result shows **31 rows passing**.

### Gram classification of MDRO reporting groups

In [3]:
%sql
select
    gram_stain_group,
    count(distinct organism_base_name) as organism_count,
    count(*) as mart_rows

from workspace.micro_dev.mart_mdro

group by gram_stain_group

order by gram_stain_group;

,gram_stain_group,organism_count,mart_rows
0,gram_negative,11,31


All **11 base organisms** represented in the current CRE/MDR mart are Gram-negative. The table contains 31 cohort–organism–phenotype combinations.

### Inspect the organism and phenotype breakdown

In [1]:
%sql
select
    organism_base_name,
    gram_stain_group,
    mdro_phenotype,
    count(*) as mart_rows

from workspace.micro_dev.mart_mdro

group by
    organism_base_name,
    gram_stain_group,
    mdro_phenotype

order by
    gram_stain_group,
    organism_base_name,
    mdro_phenotype;

,organism_base_name,gram_stain_group,mdro_phenotype,mart_rows
0,Citrobacter farmeri,gram_negative,CRE,1
1,Citrobacter freundii complex,gram_negative,CRE,3
2,Citrobacter koseri,gram_negative,CRE,3
3,Enterobacter cloacae complex,gram_negative,CRE,3
4,Escherichia coli,gram_negative,CRE,3
5,Klebsiella aerogenes,gram_negative,CRE,3
6,Klebsiella oxytoca,gram_negative,CRE,3
7,Klebsiella pneumoniae,gram_negative,CRE,3
8,Proteus mirabilis,gram_negative,CRE,3
9,Pseudomonas aeruginosa,gram_negative,MDR Pseudomonas aeruginosa,3


This checks the classification at the base-organism level, including groups that have CRE summaries but no standard antibiogram drug pairs.

### Cross-mart coverage of the Gram reference

In [7]:
%sql
with reporting_organisms as (

    select distinct
        organism_reporting_group
    from workspace.micro_dev.mart_susceptibility

    union

    select distinct
        organism_base_name as organism_reporting_group
    from workspace.micro_dev.mart_mdro

)

select
    r.organism_reporting_group

from reporting_organisms r

left join workspace.micro_dev.organism_reporting_groups g
    on r.organism_reporting_group = g.organism_reporting_group

where g.organism_reporting_group is null

order by r.organism_reporting_group;

,organism_reporting_group


Every reporting organism present in either mart should match the `organism_reporting_groups` reference seed. A passing query returns no rows.

### Cross-mart completeness

In [2]:
%sql
select
    'mart_susceptibility' as model_name,
    count(*) as row_count,
    sum(
        case
            when gram_stain_group is null then 1
            else 0
        end
    ) as missing_gram_stain_group

from workspace.micro_dev.mart_susceptibility

union all

select
    'mart_mdro' as model_name,
    count(*) as row_count,
    sum(
        case
            when gram_stain_group is null then 1
            else 0
        end
    ) as missing_gram_stain_group

from workspace.micro_dev.mart_mdro;

,model_name,row_count,missing_gram_stain_group
0,mart_susceptibility,325,0
1,mart_mdro,31,0


The saved check shows **325 susceptibility-mart rows** and **31 MDRO-mart rows**, with zero missing Gram classifications in either table.

### Result

The MDRO denominator is reconstructed independently of the mart, and Gram metadata is complete across both reporting outputs. These checks conclude the validated portion of the pipeline. The next stage is the Tableau dashboard, not further changes to the source-grain logic.